##01 - Data Profiling. Sell Out e Inventario de Refacciones

Objetivo. Realizar una exploración inicial de los datasets de ventas e invetnario para conocer su estructura, calidad y principales características antes de aplicar procesos de limpieza, transformación o análisis

En este notebook se revisan:
1. Dimensiones de los datasets
2. Columnas disponibles
3. Tipos de datos
4. Valores nulos
5. Registros duplicados
6. Cardinalidad de variables relevantes
7. Consistencia de fechas
8. Valores potencialmente atípicos o inválidos
9. Coincidencia de números de parte entre ventas e invetnario

Los resultados servirán como punto de partida para definir las reglas de limpieza y procesamineto del framework de análisis de Sell Out.

In [0]:
# Importar librerías
import pandas as pd
from pathlib import Path # Para trabajar con rutas de archivos de manera más organizada

In [0]:
# Contrucción de rutas de datasets y validación de archivos exists
DATA_PATH = Path('../data/raw')
# Lectura de dataset
sales = DATA_PATH / 'Sales_dataset.csv'
print("Sales file exists:", sales.exists())

inventory = DATA_PATH / 'Inventory_dataset.csv'
print("Inventory file exists:", inventory.exists())


In [0]:
# Carga de datos
dfSales = pd.read_csv(sales, dtype={"cd_NumParte": "string"})

dfInventory = pd.read_csv(inventory, dtype= {"cd_NumParte": "string"})

# VErificar qué se cargó
print("Ventas:")
display(dfSales.head())
print("Inventario:")
display(dfInventory.head())


In [0]:
#Conocer las dimensiones de los datasets
print("Dimensiones Ventas:", dfSales.shape)
print("Dimensiones Inventario:", dfInventory.shape)


In [0]:
#Revisar columnas, operador & representa la intersección
common_columns = set(dfSales.columns) & set(dfInventory.columns)
print("Columnas comunes:",common_columns )


In [0]:
# Primera inspección con info()
print("Información de Ventas:")
print(dfSales.info())
print("Información de Inventario:")
print(dfInventory.info())

##Obervaciones iniciales
Después de realizar la carga inicial de los datasets observé que:
- El dataset de Ventas contiene 1000 registros y 18 columnas
- El dataset de Inventario contiene 1000 registros y 25 columnas
- Ambos datasets contienen la variable cd_Id, cd_NumParte, cd_LocId y tx_Distribuidor, que podría utilizarse como punto de relación entre ventas e inventario.
- Algunas variables requieren revisión de su tipo de dato.
- Antes de integrar ambos archivos será necesario realizar pruebas de calidad

In [0]:
#Revisar valores nulos
nulls_sales = dfSales.isnull().sum()
nulls_inventory = dfInventory.isnull().sum()
print("Valores nulos en Ventas:")
print(nulls_sales)
print("Valores nulos en Inventario:")
print(nulls_inventory)

#Mostrar solo campos con valores unlos
print("Columnas con valores nulos en Sales:")
display(nulls_sales[nulls_sales > 0])
print("Columnas con valores nulos en Inventory:")
display(nulls_inventory[nulls_inventory > 0])

In [0]:
#Porcentaje de valores nulos
nulls_sales_percentage = (nulls_sales / len(dfSales)) * 100
nulls_inventory_percentage = (nulls_inventory / len(dfInventory)) * 100
display("Porcentaje de valores nulos por campo en Sales: " ,nulls_sales_percentage [nulls_sales_percentage> 0]
        .sort_values(ascending=False)
)
display("Porentaje de valores nulos por campo en Inventory: " ,nulls_inventory_percentage[nulls_inventory_percentage> 0]
        .sort_values(ascending=False)
)

In [0]:
#Buscar duplicados
duplicates_sales = dfSales.duplicated().sum()
duplicates_inventory = dfInventory.duplicated().sum()
print("Duplicados en Sales:", duplicates_sales)
print("Duplicados en Inventory:",duplicates_inventory)

#Ver los duplicados si existen
#duplicated_sales = dfSales[dfSales.duplicated(keep=False)]
#duplicated_inventory = dfInventory[dfInventory.duplicated(keep=False)]
#print("Duplicados en Sales:")
#display(duplicated_sales)
#print("Duplicados en Inventory")
#display(duplicated_inventory)

In [0]:
#Validar Nulos en campos críticos
critical_sales_columns = [
    "cd_NumParte",
    "ct_Cantidad",
    "fh_Fecha"
]
critical_inventory_columns = [
    "cd_NumParte",
    "ct_Existencia"
]

print("Nulos en campos críticos de Sales:")
display(
    dfSales[critical_sales_columns]
    .isnull()
    .sum()
)
print("Nulos en campos críticos de Inventory:")
display(
    dfInventory[critical_inventory_columns]
    .isnull()
    .sum()
)


In [0]:
# Validar Cantidades
invalid_sales_quantity = dfSales[
    dfSales["ct_Cantidad"] <= 0
]
print(
    "Ventas con cantidad menor o igual a 0"
)
display(invalid_sales_quantity)

Las cantidades en 0 dentro del data frame de Ventas, ¿es incorrecto?, no sabemos aún, podría representar:
- error
- cancelación
- transacción sin movimientos
- registro informativo
- lógica propia del sistema del Distribuidor

Lo etiquetamos como:
**Potential Data Quality Issue**

In [0]:
#Revisar cardinalidad de campos 
# Vamos a conocer cuántos valores diferentes existen
# nunique() significa, Number od unique values
print("Dealers:",dfSales["tx_Distribuidor"].nunique())
print("Produtos:",dfSales["cd_NumParte"].nunique())
print("Clientes:",dfSales["cd_IdClienteFC"].nunique())
print("Nombres de cliente:",dfSales["tx_Cliente"].nunique())


In [0]:
#Revisar  campos constantes
for col in dfSales.columns:
    unique_values = dfSales[col].nunique()
    if unique_values <= 1:
        print(f"{col}:{unique_values} valor unico")

In [0]:
#Revisar las fechas
print("Fecha mínima de ventas:")
print(dfSales["fh_Fecha"].min())

print("Fecha máxima de ventas:")
print(dfSales["fh_Fecha"].max())

Con la validación de fechas anterior, se puede determinar que, no existe una serie temporal suficiente para implementar un modelo de Pronóstico de Ventas

##Tabla Resumen

Esta tabla nos servirá para poner en práctica los elementos revisados teóricamente. Partímos de la creación de un DataFrame a partir de un diccionario de Python. 

Estamos conectando:
**diccionarios -> listas -> libfería Pandas -> EDA**

In [0]:
quality_summary = pd.DataFrame({
    "Dataset": [
        "Sales",
        "Inventory"
    ],
    "Rows" : [
        len(dfSales),
        len(dfInventory)
    ],
    "Columns" :[
        dfSales.shape[1],
        dfInventory.shape[1]
    ],
    "Missing values":[
        dfSales.isnull().sum().sum(),
        dfInventory.isnull().sum().sum()
    ],
    "Duplicate Rows": [
        dfSales.duplicated().sum(),
        dfInventory.duplicated().sum()
    ]
})

display(quality_summary)

##Conclusiones de calidad de datos

Después de realizar las validaciones iniciales se identificaron los siguientes puntos:
1. El dataset de ventas presenta **1000** **valores nulos** para el campo RFC
2. Las columnas con mayor presencia de valores nulos son:
    Columnas con **valores nulos en Sales**:
        **tx_rfc    1000**
    Columnas con **valores nulos en Inventory**:
       ** tx_Descripcion    5**
3. Se identificaron **0 registros duplicados** en ventas
4. Se identificaron 0 registros duplicados en inventario
5. Existen **46 ventas** con cantidad menor o igual a **cero**
6. El dataset contiene **1 dealer**, **1000 productos** y **793 clientes** 
7. La información disponible corresponde a **1 unica fecha de registro**


## Ejercicios

1. ¿Por qué no deberíamos hacer inmediatamente dropna()?
Porque el hecho de que el campo de un registro venga en nulo, no es argumento suficiente para discriminar el valor y asumir que no es válido

2. ¿Cuál es la diferencia entre un duplicado técnico y una venta repetida válida? Que el duplicado técnico pudo haber sido originado por un error de transacción y el de venta repetida válida es un registro correcto que puede representar dos transacciones reales

3. ¿Por qué cd_NumParte es una variable crítica?
Porque es el campo que nos servirá para ligar el dataset de Ventas con el de Inventario

4. ¿Por qué tener una sola fecha limita un modelo de predicción?
Porque no da claridad de la temporalidad de los datos, una predicción de demanda necesita aprender un comportamiento a lo largo del tiempo.

5. Si tx_Cliente tiene un solo valor pero cd_IdClienteFC tiene cientos, ¿cuál usarías para segmentar clientes y por qué? Usaría cd_IdClienteFC porque me permitirá realizar una alálisis de los datos por segmento de Clientes ya que nos permite distinguirlos